## MODEL BUILDING

In [99]:


 
# ============================================================================
#  TRAIN  MODELS 
# ============================================================================
print("="*80)
print(" TRAINING MODELS")
print("="*80)
 
models = {}
cv_scores = {}
 
# ----------------------------------------------------------------------------
# Model 1: Logistic Regression (Baseline)
# ----------------------------------------------------------------------------
 
lr_model = LogisticRegression(
    solver='saga',
    max_iter=500,  
    random_state=42,
    n_jobs=-1,
    verbose=0
)
 
lr_model.fit(X_train_balanced, y_train_balanced)
models['Logistic Regression'] = lr_model
 
# Quick 3-fold CV on subset for speed
cv_score = cross_val_score(
    lr_model, X_train_balanced, y_train_balanced, 
    cv=3, scoring='f1_macro', n_jobs=-1
)
cv_scores['Logistic Regression'] = cv_score
 
print(f"  ✓ Training complete")
print(f"  CV F1-Score (macro): {cv_score.mean():.4f} ± {cv_score.std():.4f}")
 
# ----------------------------------------------------------------------------
# Model 2: Decision Tree
# ----------------------------------------------------------------------------
print("\n Training Decision Tree...")
 
dt_model = DecisionTreeClassifier(
    max_depth=20,
    min_samples_split=100,
    min_samples_leaf=50,
    random_state=42
)
 
dt_model.fit(X_train_balanced, y_train_balanced)
models['Decision Tree'] = dt_model
 
cv_score = cross_val_score(
    dt_model, X_train_balanced, y_train_balanced, 
    cv=3, scoring='f1_macro', n_jobs=-1
)
cv_scores['Decision Tree'] = cv_score
 
print(f"  ✓ Training complete")
print(f"  CV F1-Score (macro): {cv_score.mean():.4f} ± {cv_score.std():.4f}")

 
# ----------------------------------------------------------------------------
# Model 3: Random Forest (Default)
# ----------------------------------------------------------------------------
print("\n Training Random Forest (default)...")
 
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=100,
    random_state=42,
    n_jobs=-1,
    verbose=0
)
 
rf_model.fit(X_train_balanced, y_train_balanced)
models['Random Forest'] = rf_model
 
cv_score = cross_val_score(
    rf_model, X_train_balanced, y_train_balanced, 
    cv=3, scoring='f1_macro', n_jobs=-1
)
cv_scores['Random Forest'] = cv_score
 
print(f"  ✓ Training complete")
print(f"  CV F1-Score (macro): {cv_score.mean():.4f} ± {cv_score.std():.4f}")
 
 


 


 

 TRAINING MODELS
  ✓ Training complete
  CV F1-Score (macro): 0.6488 ± 0.0299

 Training Decision Tree...
  ✓ Training complete
  CV F1-Score (macro): 0.7693 ± 0.0229

 Training Random Forest (default)...
  ✓ Training complete
  CV F1-Score (macro): 0.8002 ± 0.0053


In [ ]:
# ----------------------------------------------------------------------------
# Model 6: XGBoost (Tuned) - OPTIMIZED
# ----------------------------------------------------------------------------
y_train_xgb = y_train_balanced - 1
y_test_xgb = y_test - 1

print("Original classes:", sorted(y_train_balanced.unique()))
print("XGBoost classes:", sorted(y_train_xgb.unique()))
# OPTIMIZED: Smaller parameter grid
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [6, 10, 15],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}
 
xgb_random = RandomizedSearchCV(
    XGBClassifier(random_state=42, n_jobs=-1, verbosity=0),
    param_distributions=param_grid_xgb,
    n_iter=10,  # Reduced from 20
    cv=3,       # Reduced from 5
    scoring='f1_macro',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
 
xgb_random.fit(X_train_balanced, y_train_xgb)
 
xgb_best = xgb_random.best_estimator_
models['XGBoost (Tuned)'] = xgb_best
 
cv_score = cross_val_score(
    xgb_best, X_train_balanced, y_train_xgb, 
    cv=3, scoring='f1_macro', n_jobs=-1
)
cv_scores['XGBoost (Tuned)'] = cv_score
 
print(f"  ✓ Training complete")
print(f"  Best params: {xgb_random.best_params_}")
print(f"  CV F1-Score (macro): {cv_score.mean():.4f} ± {cv_score.std():.4f}")

Original classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
XGBoost classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Fitting 3 folds for each of 10 candidates, totalling 30 fits
  Tuning complete. Training best model on full dataset...
  ✓ Training complete
  Best params: {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 10, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
  CV F1-Score (macro): 0.8616 ± 0.0723


In [106]:
# ============================================================================
#  SAVE ALL MODELS
# ============================================================================
import joblib


print("\n" + "="*80)
print("STEP 3: SAVING MODELS")
print("="*80)
 
for model_name, model in models.items():
    filename = model_name.lower().replace(' ', '_').replace('(', '').replace(')', '') + '_model.pkl'
    joblib.dump(model, filename)
    print(f"✓ Saved: {filename}")
 
print("\n✓ All models saved successfully!")


STEP 3: SAVING MODELS
✓ Saved: random_forest_model.pkl
✓ Saved: xgboost_tuned_model.pkl
✓ Saved: decision_tree_model.pkl
✓ Saved: logistic_regression_model.pkl

✓ All models saved successfully!


In [107]:
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("MODEL TRAINING SUMMARY")
print("="*80)
 
summary_df = pd.DataFrame({
    'Model': list(cv_scores.keys()),
    'CV F1-Score (macro)': [f"{scores.mean():.4f}" for scores in cv_scores.values()],
    'CV Std Dev': [f"{scores.std():.4f}" for scores in cv_scores.values()]
})
 
print(summary_df.to_string(index=False))
 
# Find best model
best_model_name = max(cv_scores, key=lambda x: cv_scores[x].mean())
best_score = cv_scores[best_model_name].mean()
 
print(f"\n🏆 Best Model (by CV F1-Score): {best_model_name}")
print(f"   F1-Score: {best_score:.4f}")
 
print("MODEL BUILDING COMPLETE!")



MODEL TRAINING SUMMARY
              Model CV F1-Score (macro) CV Std Dev
Logistic Regression              0.6488     0.0299
      Decision Tree              0.7693     0.0229
      Random Forest              0.8002     0.0053
    XGBoost (Tuned)              0.8616     0.0723

🏆 Best Model (by CV F1-Score): XGBoost (Tuned)
   F1-Score: 0.8616
MODEL BUILDING COMPLETE!
